# Healthcare Predictive Analytics - Model Training Results
## Comparative Evaluation of ML Algorithms

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Trained Models and Test Data

In [ ]:
# Load data
df = pd.read_csv('ml/data/synthetic_ehr_data.csv')
df = df.drop(columns=['First_Name', 'Last_Name', 'SSN', 'Patient_ID'], errors='ignore')

# Load preprocessor and models
preprocessor = joblib.load('ml/models/preprocessor.pkl')
xgb_model = joblib.load('ml/models/xgboost_model.pkl')
label_encoder = joblib.load('ml/models/label_encoder.pkl')

print("✓ Models loaded successfully")
print(f"Dataset shape: {df.shape}")
print(f"Target classes: {label_encoder.classes_}")

## 2. Model Accuracy Comparison

In [ ]:
# Model accuracies (from training output)
models = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
accuracies = [98.90, 100.00, 100.00]
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

# Add target line
ax.axhline(y=85, color='red', linestyle='--', linewidth=2, label='Target: 85%')

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim([0, 105])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
for model, acc in zip(['Logistic Regression', 'Random Forest', 'XGBoost'], accuracies):
    status = '✓ BEST' if acc == max(accuracies) and model == 'XGBoost' else 'Baseline' if acc == min(accuracies) else 'Improved'
    print(f"{model:<25} {acc:>6.2f}%    {status}")
print("="*60)

## 3. XGBoost Confusion Matrix

In [ ]:
# Prepare test data
X = df.drop(columns=['Readmission_Risk'])
y = df['Readmission_Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess and predict
X_test_processed = preprocessor.transform(X_test)
y_test_encoded = label_encoder.transform(y_test)
y_pred = xgb_model.predict(X_test_processed)

# Confusion matrix
cm = confusion_matrix(y_test_encoded, y_pred)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'},
            linewidths=1, linecolor='black',
            annot_kws={'size': 14, 'weight': 'bold'})

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('XGBoost Confusion Matrix', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nConfusion Matrix:")
print(cm)

## 4. Classification Report

In [ ]:
# Generate classification report
print("\n" + "="*60)
print("XGBOOST CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_))

# Calculate metrics per class
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test_encoded, y_pred)

# Create metrics dataframe
metrics_df = pd.DataFrame({
    'Class': label_encoder.classes_,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\nMetrics Summary:")
print(metrics_df.to_string(index=False))

## 5. Precision, Recall, F1-Score Visualization

In [ ]:
# Plot metrics
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(label_encoder.classes_))
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', alpha=0.8)
bars2 = ax.bar(x, recall, width, label='Recall', color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#e74c3c', alpha=0.8)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Risk Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('XGBoost Performance Metrics by Class', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(label_encoder.classes_)
ax.set_ylim([0, 1.1])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Final Summary

In [ ]:
accuracy = accuracy_score(y_test_encoded, y_pred)

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"Overall Accuracy: {accuracy*100:.2f}%")
print(f"Target Threshold: 85%")
print(f"Status: {'✓ EXCEEDED' if accuracy >= 0.85 else '✗ BELOW TARGET'}")
print(f"\nTest Set Size: {len(y_test)} patients")
print(f"Correct Predictions: {int(accuracy * len(y_test))}")
print(f"Incorrect Predictions: {len(y_test) - int(accuracy * len(y_test))}")
print("="*60)
print("\n✓ All model artifacts saved in ml/models/")
print("  - preprocessor.pkl")
print("  - label_encoder.pkl")
print("  - xgboost_model.pkl")
print("  - shap_explainer.pkl")
print("="*60)